In [1]:
from run_single_experiment import run_experiment_1, run_experiment_2, run_experiment_3, run_experiment_4

/Users/ukun/anaconda3/envs/CTGAN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD = /Users/ukun/Desktop/captcha
captcha_data exists? True
secrets.yaml exists? True
sample type dirs: ['./captcha_data/Unusual_Detection', './captcha_data/Connect_icon', './captcha_data/Select_Animal', './captcha_data/Coordinates', './captcha_data/Geometry_Click']
{
  "providers": {
    "openai": {
      "api_key": "REDACTED_OPENAI_API_KEY"
    },
    "anthropic": {
      "api_key": "sk-ant-可选"
    },
    "gemini": {
      "api_key": "REDACTED_GEMINI_API_KEY"
    },
    "qwen": {
      "api_key": "sk-qwen-可选",
      "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1"
    }
  },
  "pricing": {
    "openai": {
      "gpt-5": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      }
    },
    "gemini": {
      "gemini-2.5-pro": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      },
      "gemini-2.5-flash": {
        "in_per_1k": 0.0003,
        "out_per_1k": 0.0025
      },
      "gemini-2.5-flash-lite": {
        "in_per_1k": 0.0001,
        "out_per

In [2]:
from pathlib import Path

RESULTS_BASE = Path("./results")
ERROR_BASE = Path("./error_analysis")

def _norm_model_name(model: str) -> str:
    return model.replace("/", "_")

def exp_results_dir(exp_name: str, provider: str, model: str) -> str:
    path = RESULTS_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)

def exp_out_csv(exp_name: str, provider: str, model: str, filename: str = "results.csv") -> str:
    return str(Path(exp_results_dir(exp_name, provider, model)) / filename)

def exp_error_dir(exp_name: str, provider: str, model: str) -> str:
    path = ERROR_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


In [3]:
SUPPORTED_TYPES = {
    "Dice_Count",
    "Geometry_Click",
    "Image_Matching",
    "Patch_Select",
    "Place_Dot",

    "Bingo",
    "Click_Order",
    "Dart_Count",
    "Image_Recognition",
    "Misleading_Click",
    "Object_Match",
    "Path_Finder",
    "Pick_Area",
    "Select_Animal",
    "Unusual_Detection",
    "Coordinates",
    "Connect_Icon",
    "Rotation_Match",
}

ERROR_TYPES = {
    "Dice_Count",
    "Click_Order",
    "Patch_Select",
    "Place_Dot"
}

In [14]:
# 实验一：Ground Truth Prompts
provider = "openai"
model = "gpt-5-chat-latest"

result1 = run_experiment_1(
    types=SUPPORTED_TYPES,
    max_per_type=10,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp1", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp1", provider, model),
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp1", provider, model),
    collect_reasoning=False,
    timeout_sec=600
)



🔵 实验一：Ground Truth Prompts
Provider: openai
Model: gpt-5-chat-latest
Thinking: False

[INFO] 将评测 180 道题（types={'Unusual_Detection', 'Image_Recognition', 'Image_Matching', 'Geometry_Click', 'Misleading_Click', 'Object_Match', 'Path_Finder', 'Bingo', 'Place_Dot', 'Coordinates', 'Dice_Count', 'Pick_Area', 'Click_Order', 'Connect_Icon', 'Rotation_Match', 'Dart_Count', 'Patch_Select', 'Select_Animal'}）


Evaluating: 100% 180/180 [16:53<00:00,  5.63s/it]


[SUMMARY by type]
- Bingo              n= 10  pass@1=0.000  avg_e2e_ms=3443.5  avg_ttft_ms=3436.1  tokens_in=7790 tokens_out=160 cost_usd=0.000000
- Click_Order        n= 10  pass@1=0.000  avg_e2e_ms=1681.5  avg_ttft_ms=1681.0  tokens_in=7140 tokens_out=600 cost_usd=0.000000
- Connect_Icon       n= 10  pass@1=0.000  avg_e2e_ms=4706.2  avg_ttft_ms=4701.9  tokens_in=21780 tokens_out=140 cost_usd=0.000000
- Coordinates        n= 10  pass@1=0.000  avg_e2e_ms=8730.4  avg_ttft_ms=8719.1  tokens_in=49700 tokens_out=140 cost_usd=0.000000
- Dart_Count         n= 10  pass@1=0.100  avg_e2e_ms=16665.3  avg_ttft_ms=16652.8  tokens_in=82490 tokens_out=140 cost_usd=0.000000
- Dice_Count         n= 10  pass@1=0.000  avg_e2e_ms=4098.1  avg_ttft_ms=4096.2  tokens_in=9630 tokens_out=130 cost_usd=0.000000
- Geometry_Click     n= 10  pass@1=0.000  avg_e2e_ms=1715.4  avg_ttft_ms=1713.9  tokens_in=4069 tokens_out=230 cost_usd=0.000000
- Image_Matching     n= 10  pass@1=0.000  avg_e2e_ms=13947.8  avg_ttft_ms

In [4]:
# 实验二：Optimized Prompts
provider = "openai"
model = "gpt-5-chat-latest"

result2 = run_experiment_2(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    max_per_type=10,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp2", provider, model),
    thinking=False,
    thinking_options={"budget": -1},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp2", provider, model),
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp2", provider, model),
    collect_reasoning=False,
    timeout_sec=600
)



🟢 实验二：Optimized Prompts
Provider: openai
Model: gpt-5-chat-latest
Prompts file: ./prompts_optimized.yaml
Thinking: False

[INFO] 将评测 180 道题（types={'Pick_Area', 'Place_Dot', 'Bingo', 'Select_Animal', 'Image_Matching', 'Dart_Count', 'Path_Finder', 'Coordinates', 'Misleading_Click', 'Unusual_Detection', 'Image_Recognition', 'Object_Match', 'Rotation_Match', 'Click_Order', 'Dice_Count', 'Patch_Select', 'Connect_Icon', 'Geometry_Click'}）


Evaluating: 100% 180/180 [16:48<00:00,  5.60s/it]


[SUMMARY by type]
- Bingo              n= 10  pass@1=0.000  avg_e2e_ms=2673.3  avg_ttft_ms=2665.8  tokens_in=8500 tokens_out=160 cost_usd=0.000000
- Click_Order        n= 10  pass@1=0.000  avg_e2e_ms=1739.3  avg_ttft_ms=1736.2  tokens_in=8490 tokens_out=600 cost_usd=0.000000
- Connect_Icon       n= 10  pass@1=0.200  avg_e2e_ms=4393.0  avg_ttft_ms=4389.2  tokens_in=22540 tokens_out=140 cost_usd=0.000000
- Coordinates        n= 10  pass@1=0.400  avg_e2e_ms=8059.0  avg_ttft_ms=8046.1  tokens_in=50470 tokens_out=140 cost_usd=0.000000
- Dart_Count         n= 10  pass@1=0.100  avg_e2e_ms=15511.3  avg_ttft_ms=15500.5  tokens_in=83510 tokens_out=140 cost_usd=0.000000
- Dice_Count         n= 10  pass@1=0.000  avg_e2e_ms=3931.0  avg_ttft_ms=3919.5  tokens_in=11820 tokens_out=130 cost_usd=0.000000
- Geometry_Click     n= 10  pass@1=0.100  avg_e2e_ms=807.6  avg_ttft_ms=806.1  tokens_in=4997 tokens_out=230 cost_usd=0.000000
- Image_Matching     n= 10  pass@1=0.000  avg_e2e_ms=17938.9  avg_ttft_ms=

In [ ]:
# 实验三：Until Correct
provider = "openai"
model = "gpt-5-chat-latest"

result3 = run_experiment_3(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    prompt_mode="auto",
    max_attempts_per_type=10,
    max_pool_per_type=10,
    out_csv=exp_out_csv("exp3", provider, model),
    provider=provider,
    model=model,
    thinking=True,
    thinking_options={"budget": -1},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp3", provider, model),
    collect_reasoning=True
)


In [ ]:
# 实验四：Few-shot
provider = "openai"
model = "gpt-5-chat-latest"

result4 = run_experiment_4(
    prompts_file="./prompts_optimized.yaml",
    few_shot_file="./few_shot_examples.yaml",
    types=ERROR_TYPES,
    max_per_type=10,
    provider=provider,
    model=model,
    n_shot=2,
    thinking=False,
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp4", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp4", provider, model),
    collect_reasoning=False,
    out_csv=exp_out_csv("exp4", provider, model),
    timeout_sec=600
    thinking=True,
    thinking_options={"budget": -1},
)



🟣 实验四：Optimized Prompts + Few-shot Learning
Provider: openai
Model: gpt-5-chat-latest
Prompts file: ./prompts_optimized.yaml
Few-shot file: ./few_shot_examples.yaml
N-shot: 2
Note: 每种类型使用前 2 个样本作为示例

[INFO] Few-shot learning enabled
[INFO] Excluding 8 few-shot examples from testing
[INFO] 将评测 32 道题（types={'Patch_Select', 'Dice_Count', 'Click_Order', 'Place_Dot'}）


Evaluating: 100% 32/32 [02:22<00:00,  4.44s/it]


[SUMMARY by type]
- Click_Order        n=  8  pass@1=0.000  avg_e2e_ms=2825.5  avg_ttft_ms=2822.7  tokens_in=21176 tokens_out=480 cost_usd=0.000000
- Dice_Count         n=  8  pass@1=0.125  avg_e2e_ms=8446.3  avg_ttft_ms=8438.9  tokens_in=24112 tokens_out=104 cost_usd=0.000000
- Patch_Select       n=  8  pass@1=0.000  avg_e2e_ms=3787.6  avg_ttft_ms=3787.2  tokens_in=13197 tokens_out=229 cost_usd=0.000000
- Place_Dot          n=  8  pass@1=0.000  avg_e2e_ms=2685.9  avg_ttft_ms=2681.7  tokens_in=18024 tokens_out=184 cost_usd=0.000000
[INFO] Token summary written to results/exp4/openai/gpt-5-chat-latest/exp4_fewshot_openai_gpt-5-chat-latest_token_summary.json

[OVERALL] tasks=32  pass@1=0.031  sum_e2e_ms=141962.6  wall_ms=142030.2
[OVERALL TOKENS] tokens_in=76509 tokens_out=997 cost_usd=0.000000
[DONE] Pass@1 = 1/32 = 0.031 ; errors=0. 结果已保存到 results/exp4/openai/gpt-5-chat-latest/results.csv
✅ 错误分析已保存到: error_analysis/exp4/openai/gpt-5-chat-latest/exp4_fewshot
   - 错误案例: error_analysis/e